In [6]:
import os
#
import requests
import json
import pandas as pd
import pytz
from datetime import datetime, timedelta
#
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows
from collections import defaultdict
import hashlib

In [7]:
output_file = "C:\\Users\\KDrogaieva\\OneDrive - Learning.com\\EDW\\tableau_schedule.xlsx"

In [ ]:
server_url = "https://10ay.online.tableau.com/"
site_name = "lcomreporting"


personal_access_token_name = os.environ.get("Script_Token_Name") # Script
personal_access_token_secret = os.environ.get("Script_Token_Secret")


In [9]:
num_slots = 2
window_start_str = "03:00"
window_end_str   = "08:00"

In [10]:
# Authenticate and get token
auth_url = f"{server_url}/api/3.18/auth/signin"
auth_payload = {
    "credentials": {
        "personalAccessTokenName": personal_access_token_name,
        "personalAccessTokenSecret": personal_access_token_secret,
        "site": {"contentUrl": site_name}
    }
}
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

response = requests.post(auth_url, json=auth_payload, headers=headers)
response.raise_for_status()

# Print response content for debugging
print("Response content:", response.text)



Response content: {"credentials":{"site":{"id":"11fe3bde-d66a-4e82-9be1-79a1a3305f29","contentUrl":"lcomreporting"},"user":{"id":"a2586edc-7f08-451b-b5a2-a2c6284f131f"},"token":"GUsmWDJRSlKfic9FUNErOw|5woMSIppyGWXuAcrZOybTnltgxjByavC|11fe3bde-d66a-4e82-9be1-79a1a3305f29","estimatedTimeToExpiration":"179:23:57"}}


In [11]:
# Only try to parse JSON if response is not empty
if response.text:
    token = response.json()['credentials']['token']
    site_id = response.json()['credentials']['site']['id']
else:
    raise ValueError("Empty response received from Tableau authentication endpoint.")

In [12]:
def get_all_connection_types(token, site_id, datasource_ids):
    conn_map = {}
    for ds_id in datasource_ids:
        conn_type = get_datasource_connection_type(token, site_id, ds_id)
        conn_map[ds_id] = conn_type
    return conn_map
#
def get_datasource_connection_type(token, site_id, datasource_id):
    url = f"{server_url}/api/3.21/sites/{site_id}/datasources/{datasource_id}/connections"
    headers = {"X-Tableau-Auth": token, "Accept": "application/json"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    data = response.json()
    connections = data.get("connections", {}).get("connection", [])
    if connections:
        conn_type = connections[0].get("type", "<missing>")  # ✅ FIXED HERE
        return conn_type
    else:
        return "<no-connection>"
#
def get_users(token, site_id):
    url = f"{server_url}/api/3.21/sites/{site_id}/users?pageSize=1000"
    headers = {"X-Tableau-Auth": token, "Accept": "application/json"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    users = response.json()['users']['user']
    return {u['id']: u['name'] for u in users}  # or use u['fullName'] if needed
#
def get_projects(token, site_id):
    url = f"{server_url}/api/3.21/sites/{site_id}/projects?pageSize=1000"
    headers = {"X-Tableau-Auth": token, "Accept": "application/json"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    project_list = response.json()['projects']['project']

    return {
        proj['id']: {
            'name': proj['name'],
            'parent_id': proj.get('parentProjectId')
        }
        for proj in project_list
    }
#
def build_project_paths(projects):
    project_paths = {}

    def resolve_path(proj_id):
        if proj_id is None:
            return ""
        if proj_id in project_paths:
            return project_paths[proj_id]
        proj = projects.get(proj_id)
        if not proj:
            return "<unknown>"
        parent_path = resolve_path(proj.get("parent_id"))
        full_path = f"{parent_path}/{proj['name']}" if parent_path else proj['name']
        project_paths[proj_id] = full_path
        return full_path

    # Resolve all
    for proj_id in projects.keys():
        resolve_path(proj_id)

    return project_paths
#

def get_datasources(token, site_id):
    url = f"{server_url}/api/3.21/sites/{site_id}/datasources?pageSize=1000"
    headers = {"X-Tableau-Auth": token, "Accept": "application/json"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    datasources = response.json()['datasources']['datasource']
    return {
        ds['id']: {
            'name': ds['name'],
            'tags':  ds['tags'],
            'project_id': ds.get('project', {}).get('id'),
            'owner_id': ds.get('owner', {}).get('id')
        }
        for ds in datasources
    },response.json()
#
def get_extract_refresh_tasks(token, site_id):
    url = f"{server_url}/api/3.21/sites/{site_id}/tasks/extractRefreshes"
    headers = {
        "X-Tableau-Auth": token,
        "Accept": "application/json"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()['tasks']['task'],response.json()

In [13]:
ds_id_to_name,ds_response = get_datasources(token, site_id)

In [14]:
tasks,response = get_extract_refresh_tasks(token, site_id)

In [15]:
projects = get_projects(token, site_id)
project_paths = build_project_paths(projects)

In [16]:
users = get_users(token, site_id)

In [17]:
conn_types = get_all_connection_types(token, site_id, ds_id_to_name.keys())

In [18]:
rows = []
for task in tasks:
    er = task['extractRefresh']
    ds_id = er['datasource']['id']
    ds_name = "<unknown>"
    project_name = "<unknown>"

    # Try to pull ds info if it exists
    ds_info = ds_id_to_name.get(ds_id, {})
    ds_name = ds_info.get('name', '<unknown>')
    s = ds_info.get('tags', {})
    if 'tag' in s and isinstance(s['tag'], list) and s['tag']:
        ds_tag = ds_info.get('tags', '')['tag'][0]['label'].replace('Priority:', '').strip()
    else:
        ds_tag = ''
    owner_id = ds_info.get('owner_id')
    owner = users.get(owner_id, '<unknown>')

    project_id = ds_info.get('project_id') or er['datasource'].get('project', {}).get('id')
    project_path = project_paths.get(project_id, '<unknown>')

    connection_type = conn_types.get(ds_id, '<unknown>')

    

    if connection_type == "redshift":
        source_group = "Redshift via Bridge"
    elif connection_type == "salesforce":
        source_group = "Salesforce"
    else:
        source_group = "Other"

    rows.append({
    "Priority": ds_tag,
    "Datasource Name": ds_name,
    "Owner": owner,
    "Project Path": project_path,
    "Connection Type": source_group,
    "Schedule Frequency": er.get("schedule", {}).get("frequency"),
    "Start Time": er.get("schedule", {}).get("frequencyDetails", {}).get("start"),
    "Next Run": er.get("schedule", {}).get("nextRunAt")
})

df = pd.DataFrame(rows)

df['Next Run'] = pd.to_datetime(df['Next Run'], utc=True, errors='coerce')
df['Next Run PST'] = df['Next Run'].dt.tz_convert('US/Pacific')

In [19]:
df = df.sort_values(by='Next Run').reset_index(drop=True)

In [20]:
df

,Priority,Datasource Name,Owner,Project Path,Connection Type,Schedule Frequency,Start Time,Next Run,Next Run PST
0,,champion_teachers,abutler@learning.com,Sandbox/Data Sources,Redshift via Bridge,Daily,17:50:00,2026-08-25 00:50:00+00:00,2026-08-24 17:50:00-07:00
1,,skillscheck_schoolyear,abutler@learning.com,Sandbox/Data Sources,Redshift via Bridge,Daily,17:55:00,2026-08-25 00:55:00+00:00,2026-08-24 17:55:00-07:00
2,,SFDC Monthly Booking,kdrogaieva@learning.com,Revenue/Data Sources/Monthly Booking Reconcili...,Salesforce,Daily,19:30:00,2026-08-25 02:30:00+00:00,2026-08-24 19:30:00-07:00
3,,SFDC Account,kdrogaieva@learning.com,Reporting & Analytics/Data Sources/SFDC-Fivetr...,Salesforce,Daily,19:35:00,2026-08-25 02:35:00+00:00,2026-08-24 19:35:00-07:00
4,,SFDC User,kdrogaieva@learning.com,Reporting & Analytics/Data Sources/SFDC-Fivetr...,Salesforce,Daily,19:35:00,2026-08-25 02:35:00+00:00,2026-08-24 19:35:00-07:00
5,,SFDC Product,kdrogaieva@learning.com,Reporting & Analytics/Data Sources/SFDC-Fivetr...,Salesforce,Daily,19:35:00,2026-08-25 02:35:00+00:00,2026-08-24 19:35:00-07:00
6,,SFDC Opportunity,kdrogaieva@learning.com,Reporting & Analytics/Data Sources/SFDC-Fivetr...,Salesforce,Daily,19:35:00,2026-08-25 02:35:00+00:00,2026-08-24 19:35:00-07:00
7,,SFDC Case,kdrogaieva@learning.com,Reporting & Analytics/Data Sources/SFDC-Fivetr...,Salesforce,Daily,19:35:00,2026-08-25 02:35:00+00:00,2026-08-24 19:35:00-07:00
8,,SFDC Opportunity Line Item,kdrogaieva@learning.com,Reporting & Analytics/Data Sources/SFDC-Fivetr...,Salesforce,Daily,19:35:00,2026-08-25 02:35:00+00:00,2026-08-24 19:35:00-07:00
9,,Fivetran Opportunity,kdrogaieva@learning.com,Reporting & Analytics/Data Sources/SFDC-Fivetr...,Redshift via Bridge,Daily,19:40:00,2026-08-25 02:40:00+00:00,2026-08-24 19:40:00-07:00


In [21]:
#removing timezone for Excel
for col in ["Next Run", "Next Run PST"]:
    if col in df.columns:
        # Convert to naive datetime (remove tzinfo)
        df[col] = pd.to_datetime(df[col]).dt.tz_localize(None)

In [22]:
#pastel_colors = ['FFDDC1', 'C1E1C1', 'C1D4FF', 'F5C1FF', 'FFF7C1', 'FFD1DC']
pastel_colors = ['D0E8F2', 'B0D9E8', 'A0CBE2', '90BDDC', 'A9D3EA', 'BFE3F2']
fills = [PatternFill(start_color=color, end_color=color, fill_type="solid") for color in pastel_colors]
fill_map = dict()

In [23]:
group_counts = df.groupby(['Connection Type', 'Next Run']).size()
duplicate_groups = set(group_counts[group_counts > 1].index)

wb = Workbook()
ws = wb.active
ws.title = "Data"

header_font = Font(bold=True)
header_alignment = Alignment(horizontal='center', vertical='center')
border = Border(
    left=Side(style='thin'),
    right=Side(style='thin'),
    top=Side(style='thin'),
    bottom=Side(style='thin')
)

# Write data and style cells
for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), start=1):
    for c_idx, value in enumerate(row, start=1):
        cell = ws.cell(row=r_idx, column=c_idx, value=value)
        cell.border = border

        if r_idx == 1:
            cell.font = header_font
            cell.alignment = header_alignment
        elif c_idx == 1:  # Center first column (all rows except header)
            cell.alignment = Alignment(horizontal='center', vertical='center')

# Apply AutoFilter to header row
ws.auto_filter.ref = ws.dimensions

# Apply pastel highlight to groups with same (Connection Type, Next Run)
group_color_index = 0



for row_idx in range(2, ws.max_row + 1):
    conn_type = ws.cell(row=row_idx, column=5).value
    next_run = pd.to_datetime(ws.cell(row=row_idx, column=8).value)
    group_key = (conn_type, next_run)

    if group_key in duplicate_groups:
        if group_key not in fill_map:
            fill_map[group_key] = fills[group_color_index % len(fills)]
            group_color_index += 1
        fill = fill_map[group_key]
        for col_idx in range(1, ws.max_column + 1):
            ws.cell(row=row_idx, column=col_idx).fill = fill

# Set column widths (tune as needed)
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        if cell.value:
            max_length = max(max_length, len(str(cell.value)))
    adjusted_width = max(max_length + 2, 20)  # ensure datetime fits
    ws.column_dimensions[col_letter].width = adjusted_width

In [24]:

wb.save(output_file)
wb.close()
del wb 

In [25]:
def _to_time_series(s: pd.Series) -> pd.Series:
    """
    Convert 'HH:MM:SS' strings into datetime.time objects.
    Example: "19:35:00" -> datetime.time(19, 35, 0)
    """
    return s.astype(str).apply(
        lambda x: datetime.strptime(x, "%H:%M:%S").time() if pd.notna(x) else None
    )

def _prep_structs(df: pd.DataFrame):
    """Prepare helper structures used by both capacity and slot selection."""
    work = df.copy()
    work["Start Time"] = _to_time_series(work["Start Time"])

    # (1) Counts for 'Redshift via Bridge' & Daily at each minute
    mask_bridge_daily = (
        (work["Connection Type"] == "Redshift via Bridge")
        & (work["Schedule Frequency"].astype(str).str.lower() == "daily")
    )
    bridge_counts = (
        work.loc[mask_bridge_daily, "Start Time"]
            .value_counts(dropna=False)
            .to_dict()
    )

    # (3) All scheduled times for neighbor check (any type/frequency)
    all_times = set(work["Start Time"].dropna().tolist())
    return bridge_counts, all_times

def _generate_candidates():
    """5-minute boundary candidates strictly inside the [start, end) window."""
    ws = datetime.strptime(window_start_str, "%H:%M")
    we = datetime.strptime(window_end_str, "%H:%M")
    cur = ws + timedelta(minutes=5)  # strictly AFTER ws
    out = []
    while cur < we:  # strictly BEFORE we
        out.append(cur.time())
        cur += timedelta(minutes=5)
    return out

def _is_valid(slot_time, bridge_counts, all_times):
    # Rule (1): ≤ 2 bridge-daily items already scheduled at this minute
    if bridge_counts.get(slot_time, 0) > 2:
        return False
    # Rule (3): no item exactly 5 minutes before or after (any type/frequency)
    dt = datetime(2000, 1, 1, slot_time.hour, slot_time.minute)
    before = (dt - timedelta(minutes=5)).time()
    after  = (dt + timedelta(minutes=5)).time()
    if before in all_times or after in all_times:
        return False
    return True

def get_slot_capacity(df: pd.DataFrame):
    """
    Returns overall capacity stats inside the window:
      - total:   total 5-min boundary slots strictly inside (start, end)
      - occupied:total - free (i.e., blocked by your rules)
      - free:    number of slots that pass the validity rules
      - free_slots: list of "HH:MM" strings for valid slots (for convenience)
    """
    bridge_counts, all_times = _prep_structs(df)
    candidates = _generate_candidates()

    free_times = [
        t for t in candidates
        if _is_valid(t, bridge_counts, all_times)
    ]

    total = len(candidates)
    free = len(free_times)
    occupied = total - free

    free_slots = [f"{t.hour:02d}:{t.minute:02d}" for t in free_times]
    return {
        "total": total,
        "occupied": occupied,
        "free": free,
        "free_slots": free_slots,
    }

def find_available_slots(df: pd.DataFrame) -> list:
    """
    Returns up to `num_slots` valid start times as ['HH:MM', ...],
    using the same rules and current globals.
    """
    bridge_counts, all_times = _prep_structs(df)
    candidates = _generate_candidates()

    results = []
    for t in candidates:
        if _is_valid(t, bridge_counts, all_times):
            results.append(f"{t.hour:02d}:{t.minute:02d}")
            if len(results) >= num_slots:
                break
    return results

In [26]:
cap = get_slot_capacity(df)
print(f"{'Total Slots':<15}{'Occupied':<15}{'Free':<15}")
print(f"{cap['total']:<15}{cap['occupied']:<15}{cap['free']:<15}")

Total Slots    Occupied       Free           
59             30             29             


In [27]:

slots = find_available_slots(df)
print(slots)

['04:50', '05:00']
